# Experiment Grid — Vacancies 2026
Runs multiple configs, saves OOF MAPE + submission for each.

| Config | name_clean TE | num_leaves | Models |
|--------|--------------|------------|--------|
| A | No  | 255 | LGB+XGB |
| B | No  | 511 | LGB+XGB |
| C | Yes | 255 | LGB+XGB |
| D | No  | 511 | XGB only |
| E | No  | 511 | LGB only |

### 1. Imports & Shared Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

SEED     = 42
N_SPLITS = 5
STE_K    = 20
np.random.seed(SEED)

train_raw = pd.read_csv('data/train.csv')
test_raw  = pd.read_csv('data/test_x.csv')
test_ids  = test_raw['id'].values
print(f'train: {train_raw.shape}, test: {test_raw.shape}')

train: (49051, 26), test: (12263, 25)


### 2. Text Features (shared across all configs)

In [2]:
DESC_COL  = 'lemmaized_wo_stopwords_raw_description'
TITLE_COL = 'name_clean'

for df in [train_raw, test_raw]:
    df[DESC_COL]  = df[DESC_COL].fillna('')
    df[TITLE_COL] = df[TITLE_COL].fillna('')

n_train = len(train_raw)
corpus_desc  = pd.concat([train_raw[DESC_COL],  test_raw[DESC_COL]],  ignore_index=True)
corpus_title = pd.concat([train_raw[TITLE_COL], test_raw[TITLE_COL]], ignore_index=True)

SVD_DESC, SVD_TITLE = 100, 40

tfidf_desc  = TfidfVectorizer(max_features=6000, sublinear_tf=True, min_df=3, ngram_range=(1, 2), dtype=np.float32)
tfidf_title = TfidfVectorizer(max_features=3000, sublinear_tf=True, min_df=2, ngram_range=(1, 2), dtype=np.float32)
svd_desc    = TruncatedSVD(n_components=SVD_DESC,  random_state=SEED)
svd_title   = TruncatedSVD(n_components=SVD_TITLE, random_state=SEED)

mat_desc  = svd_desc.fit_transform(tfidf_desc.fit_transform(corpus_desc)).astype(np.float32)
mat_title = svd_title.fit_transform(tfidf_title.fit_transform(corpus_title)).astype(np.float32)

desc_cols  = [f'desc_svd_{i}'  for i in range(SVD_DESC)]
title_cols = [f'title_svd_{i}' for i in range(SVD_TITLE)]

SVD_TRAIN = pd.DataFrame(np.hstack([mat_desc[:n_train], mat_title[:n_train]]), columns=desc_cols + title_cols)
SVD_TEST  = pd.DataFrame(np.hstack([mat_desc[n_train:], mat_title[n_train:]]), columns=desc_cols + title_cols)

print(f'SVD matrices ready: {SVD_TRAIN.shape}')

SVD matrices ready: (49051, 140)


### 3. Core Feature Engineering (shared)

In [3]:
EXPERIENCE_ORDER = ['Нет опыта', 'От 1 года до 3 лет', 'От 3 до 6 лет', 'Более 6 лет']

DROP_COLS = [
    'id', 'salary_mean_net',
    'raw_description', 'raw_branded_description',
    'lemmaized_wo_stopwords_raw_branded_description',
    'lemmaized_wo_stopwords_raw_description',
    'name', 'name_clean', 'unified_address_country',
]

LOW_CARD_COLS = [
    'schedule_name', 'employment_name',
    'is_branded_description', 'if_foreign_language',
    'accept_handicapped', 'accept_kids',
]

BASE_TE_COLS = [
    'employer_id', 'employer_name',
    'professional_roles_name', 'specializations_profarea_name',
    'employer_industries',
    'unified_address_city', 'unified_address_state', 'unified_address_region',
]


def prepare_data(use_name_clean_te: bool):
    train = train_raw.copy()
    test  = test_raw.copy()

    te_cols = BASE_TE_COLS + (['name_clean'] if use_name_clean_te else [])

    exp_map = {v: i for i, v in enumerate(EXPERIENCE_ORDER)}
    for df in [train, test]:
        df['experience_ord'] = df['experience_name'].map(exp_map).fillna(-1).astype(np.int8)
        df['desc_word_count'] = df[DESC_COL].str.split().str.len().fillna(0).astype(np.int16)
        df['has_skills']    = (df['key_skills_name'].fillna('[]') != '[]').astype(np.int8)
        df['has_languages'] = (df['languages_name'].fillna('[]') != '[]').astype(np.int8)
        df['skills_count']  = df['key_skills_name'].fillna('[]').str.count(',').astype(np.int16)
        city = df['unified_address_city'].fillna('').str.lower()
        df['is_moscow'] = city.str.contains('москва').astype(np.int8)
        df['is_spb']    = city.str.contains('санкт').astype(np.int8)
        for col in te_cols:
            if col in df.columns:
                df[col] = df[col].fillna('__NA__').astype(str)

    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.float32)
    train[LOW_CARD_COLS] = oe.fit_transform(train[LOW_CARD_COLS].astype(str))
    test[LOW_CARD_COLS]  = oe.transform(test[LOW_CARD_COLS].astype(str))

    y_raw = train['salary_mean_net'].values.astype(np.float64)
    y     = np.log1p(y_raw).astype(np.float32)

    base_cols = [
        c for c in train.columns
        if c not in DROP_COLS + te_cols + ['experience_name', 'key_skills_name', 'languages_name']
        and train[c].dtype != object
    ]

    X_base_tr = train[base_cols].reset_index(drop=True)
    X_base_te = test[[c for c in base_cols if c in test.columns]].reset_index(drop=True)
    X_base_te = X_base_te.reindex(columns=X_base_tr.columns)

    return train, test, y_raw, y, X_base_tr, X_base_te, te_cols


def smoothed_te(col_tr, col_val, col_te, y_tr, global_mean, k=STE_K):
    stats = pd.DataFrame({'y': y_tr, 'cat': col_tr.values}).groupby('cat')['y'].agg(['mean', 'count'])
    stats['smooth'] = (stats['count'] * stats['mean'] + k * global_mean) / (stats['count'] + k)
    te_map = stats['smooth']
    return (
        col_tr.map(te_map).fillna(global_mean),
        col_val.map(te_map).fillna(global_mean),
        col_te.map(te_map).fillna(global_mean),
    )


def build_fold(train, test, X_base_tr, X_base_te, te_cols, tr_idx, val_idx, y, global_mean):
    te_tr_l, te_val_l, te_te_l = [], [], []
    for col in te_cols:
        s_tr, s_val, s_te = smoothed_te(
            train[col].iloc[tr_idx].reset_index(drop=True),
            train[col].iloc[val_idx].reset_index(drop=True),
            test[col].reset_index(drop=True),
            y[tr_idx], global_mean,
        )
        te_tr_l.append(s_tr.rename(f'te_{col}'))
        te_val_l.append(s_val.rename(f'te_{col}'))
        te_te_l.append(s_te.rename(f'te_{col}'))

    te_tr  = pd.concat(te_tr_l,  axis=1)
    te_val = pd.concat(te_val_l, axis=1)
    te_te  = pd.concat(te_te_l,  axis=1)

    def assemble(base_idx, svd_df, te):
        if base_idx is not None:
            base = X_base_tr.iloc[base_idx].reset_index(drop=True)
            svd  = svd_df.iloc[base_idx].reset_index(drop=True)
        else:
            base = X_base_te.reset_index(drop=True)
            svd  = SVD_TEST.reset_index(drop=True)
        out = pd.concat([base, svd, te], axis=1)
        out['exp_x_role_te'] = out['experience_ord'].astype(float) * out['te_professional_roles_name']
        out['moscow_x_exp']  = out['is_moscow'].astype(float) * out['experience_ord'].astype(float)
        out['skills_x_lang'] = out['skills_count'].astype(float) * out['has_languages'].astype(float)
        if 'te_name_clean' in out.columns:
            out['exp_x_title_te'] = out['experience_ord'].astype(float) * out['te_name_clean']
        return out

    return (
        assemble(tr_idx,  SVD_TRAIN, te_tr),
        assemble(val_idx, SVD_TRAIN, te_val),
        assemble(None,    None,      te_te),
    )

print('Helper functions ready.')

Helper functions ready.


### 4. Experiment Runner

In [4]:
def run_lgb(X_tr, y_tr, X_val, y_val, num_leaves):
    params = {
        'objective': 'regression', 'metric': 'rmse',
        'n_estimators': 8000, 'learning_rate': 0.02,
        'num_leaves': num_leaves, 'min_child_samples': 15,
        'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5,
        'reg_alpha': 0.05, 'reg_lambda': 0.1,
        'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(99999)])
    return m


def run_xgb(X_tr, y_tr, X_val, y_val):
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse',
        'n_estimators': 8000, 'learning_rate': 0.05,
        'max_depth': 7, 'min_child_weight': 10,
        'subsample': 0.8, 'colsample_bytree': 0.6,
        'reg_alpha': 0.05, 'reg_lambda': 0.1,
        'random_state': SEED, 'n_jobs': -1,
        'tree_method': 'hist', 'verbosity': 0,
        'early_stopping_rounds': 200,
    }
    m = xgb.XGBRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return m


def run_experiment(name, use_name_clean_te, num_leaves, use_lgb=True, use_xgb=True):
    print(f'\n{"="*60}')
    print(f'CONFIG {name}: name_clean_te={use_name_clean_te}, num_leaves={num_leaves}, lgb={use_lgb}, xgb={use_xgb}')
    print(f'{"="*60}')

    train, test, y_raw, y, X_base_tr, X_base_te, te_cols = prepare_data(use_name_clean_te)
    kf          = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    global_mean = float(np.mean(y))

    oof_lgb  = np.zeros(len(train)) if use_lgb else None
    oof_xgb  = np.zeros(len(train)) if use_xgb else None
    pred_lgb = np.zeros(len(test))  if use_lgb else None
    pred_xgb = np.zeros(len(test))  if use_xgb else None

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_base_tr)):
        X_tr, X_val, X_te = build_fold(train, test, X_base_tr, X_base_te, te_cols,
                                        tr_idx, val_idx, y, global_mean)
        fold_info = f'Fold {fold+1}'

        if use_lgb:
            m = run_lgb(X_tr, y[tr_idx], X_val, y[val_idx], num_leaves)
            vp = np.expm1(m.predict(X_val))
            oof_lgb[val_idx] = vp
            pred_lgb += np.expm1(m.predict(X_te)) / N_SPLITS
            mape_l = np.mean(np.abs((y_raw[val_idx] - vp) / (y_raw[val_idx] + 1e-8)))
            fold_info += f' | LGB: {mape_l:.4f} (iter {m.best_iteration_})'

        if use_xgb:
            m = run_xgb(X_tr, y[tr_idx], X_val, y[val_idx])
            vp = np.expm1(m.predict(X_val))
            oof_xgb[val_idx] = vp
            pred_xgb += np.expm1(m.predict(X_te)) / N_SPLITS
            mape_x = np.mean(np.abs((y_raw[val_idx] - vp) / (y_raw[val_idx] + 1e-8)))
            fold_info += f' | XGB: {mape_x:.4f} (iter {m.best_iteration})'

        print(fold_info)

    results = {}
    preds_list = []
    weights    = []

    if use_lgb:
        m_lgb = np.mean(np.abs((y_raw - oof_lgb) / (y_raw + 1e-8)))
        results['lgb_oof'] = m_lgb
        preds_list.append(pred_lgb)
        weights.append(1.0 / m_lgb)
        print(f'LGB OOF MAPE: {m_lgb:.4f}')

    if use_xgb:
        m_xgb = np.mean(np.abs((y_raw - oof_xgb) / (y_raw + 1e-8)))
        results['xgb_oof'] = m_xgb
        preds_list.append(pred_xgb)
        weights.append(1.0 / m_xgb)
        print(f'XGB OOF MAPE: {m_xgb:.4f}')

    w = np.array(weights)
    pred_blend = sum(p * wi for p, wi in zip(preds_list, w)) / w.sum()

    if len(preds_list) > 1:
        oof_blend = sum(o * wi for o, wi in zip(
            [oof_lgb if use_lgb else None, oof_xgb if use_xgb else None], w
        ) if o is not None) / w.sum()
        blend_mape = np.mean(np.abs((y_raw - oof_blend) / (y_raw + 1e-8)))
        results['ensemble_oof'] = blend_mape
        print(f'Ensemble OOF MAPE: {blend_mape:.4f}')

    final_preds = np.clip(pred_blend, 0, None)
    fname = f'submission_exp_{name}.csv'
    pd.DataFrame({'id': test_ids, 'salary_mean_net': final_preds}).to_csv(fname, index=False)
    print(f'Saved: {fname}')

    return results

print('Runner ready.')

Runner ready.


### 5. Run All Configs

In [5]:
all_results = {}

# Config A: baseline (no name_clean TE, num_leaves=255, LGB+XGB)
all_results['A'] = run_experiment('A', use_name_clean_te=False, num_leaves=255, use_lgb=True, use_xgb=True)


CONFIG A: name_clean_te=False, num_leaves=255, lgb=True, xgb=True
Fold 1 | LGB: 0.2744 (iter 1400) | XGB: 0.2676 (iter 1496)
Fold 2 | LGB: 0.2646 (iter 3369) | XGB: 0.2573 (iter 3045)
Fold 3 | LGB: 0.2704 (iter 2222) | XGB: 0.2660 (iter 2062)
Fold 4 | LGB: 0.2661 (iter 2006) | XGB: 0.2586 (iter 2048)
Fold 5 | LGB: 0.2636 (iter 2222) | XGB: 0.2594 (iter 1813)
LGB OOF MAPE: 0.2679
XGB OOF MAPE: 0.2618
Ensemble OOF MAPE: 0.2638
Saved: submission_exp_A.csv


In [6]:
# Config B: deeper LGB (num_leaves=511, no name_clean TE)
all_results['B'] = run_experiment('B', use_name_clean_te=False, num_leaves=511, use_lgb=True, use_xgb=True)


CONFIG B: name_clean_te=False, num_leaves=511, lgb=True, xgb=True
Fold 1 | LGB: 0.2717 (iter 1190) | XGB: 0.2676 (iter 1496)
Fold 2 | LGB: 0.2647 (iter 2559) | XGB: 0.2573 (iter 3045)
Fold 3 | LGB: 0.2704 (iter 1594) | XGB: 0.2660 (iter 2062)
Fold 4 | LGB: 0.2654 (iter 1527) | XGB: 0.2586 (iter 2048)
Fold 5 | LGB: 0.2619 (iter 1317) | XGB: 0.2594 (iter 1813)
LGB OOF MAPE: 0.2668
XGB OOF MAPE: 0.2618
Ensemble OOF MAPE: 0.2632
Saved: submission_exp_B.csv


In [7]:
# Config C: with name_clean TE (num_leaves=255)
all_results['C'] = run_experiment('C', use_name_clean_te=True, num_leaves=255, use_lgb=True, use_xgb=True)


CONFIG C: name_clean_te=True, num_leaves=255, lgb=True, xgb=True
Fold 1 | LGB: 0.3041 (iter 1200) | XGB: 0.3002 (iter 1361)
Fold 2 | LGB: 0.2988 (iter 1266) | XGB: 0.2977 (iter 1714)
Fold 3 | LGB: 0.3031 (iter 1374) | XGB: 0.2993 (iter 1350)
Fold 4 | LGB: 0.3004 (iter 1144) | XGB: 0.2985 (iter 1323)
Fold 5 | LGB: 0.2983 (iter 1196) | XGB: 0.2978 (iter 1728)
LGB OOF MAPE: 0.3010
XGB OOF MAPE: 0.2987
Ensemble OOF MAPE: 0.2992
Saved: submission_exp_C.csv


In [8]:
# Config D: XGB only (num_leaves irrelevant)
all_results['D'] = run_experiment('D', use_name_clean_te=False, num_leaves=255, use_lgb=False, use_xgb=True)


CONFIG D: name_clean_te=False, num_leaves=255, lgb=False, xgb=True
Fold 1 | XGB: 0.2676 (iter 1496)
Fold 2 | XGB: 0.2573 (iter 3045)
Fold 3 | XGB: 0.2660 (iter 2062)
Fold 4 | XGB: 0.2586 (iter 2048)
Fold 5 | XGB: 0.2594 (iter 1813)
XGB OOF MAPE: 0.2618
Saved: submission_exp_D.csv


In [9]:
# Config E: LGB only (num_leaves=511)
all_results['E'] = run_experiment('E', use_name_clean_te=False, num_leaves=511, use_lgb=True, use_xgb=False)


CONFIG E: name_clean_te=False, num_leaves=511, lgb=True, xgb=False
Fold 1 | LGB: 0.2717 (iter 1190)
Fold 2 | LGB: 0.2647 (iter 2559)
Fold 3 | LGB: 0.2704 (iter 1594)
Fold 4 | LGB: 0.2654 (iter 1527)
Fold 5 | LGB: 0.2619 (iter 1317)
LGB OOF MAPE: 0.2668
Saved: submission_exp_E.csv


### 6. Results Summary

In [10]:
print('\n=== RESULTS SUMMARY ===')
print(f'{"Config":<8} {"LGB OOF":<12} {"XGB OOF":<12} {"Ensemble OOF":<14} {"Submission file"}')
print('-' * 70)
for cfg, res in all_results.items():
    lgb_s = f"{res.get('lgb_oof', '-'):.4f}" if 'lgb_oof' in res else '-'
    xgb_s = f"{res.get('xgb_oof', '-'):.4f}" if 'xgb_oof' in res else '-'
    ens_s = f"{res.get('ensemble_oof', '-'):.4f}" if 'ensemble_oof' in res else lgb_s if 'lgb_oof' in res else xgb_s
    print(f'{cfg:<8} {lgb_s:<12} {xgb_s:<12} {ens_s:<14} submission_exp_{cfg}.csv')


=== RESULTS SUMMARY ===
Config   LGB OOF      XGB OOF      Ensemble OOF   Submission file
----------------------------------------------------------------------
A        0.2679       0.2618       0.2638         submission_exp_A.csv
B        0.2668       0.2618       0.2632         submission_exp_B.csv
C        0.3010       0.2987       0.2992         submission_exp_C.csv
D        -            0.2618       0.2618         submission_exp_D.csv
E        0.2668       -            0.2668         submission_exp_E.csv
